# Basic Pulse Scheduler for TProc V2
1-16 tone per channel pulse scheduler.

## Initialization

In [1]:
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_216.bit", "qick_216.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_216.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_216.hwh"))

print("Native Linux symlinks successfully recreated!")

Native Linux symlinks successfully recreated!


In [3]:
from qick import QickSoc

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16/out/qick_216.bit", force_init_clks=True)

soccfg = soc

# Print to check soc details and show code has run
print(soc)

configuring reference clock chips, as requested
LMK04828 clock reference = 245.760 MHz, LMX2594 clock synth = 491.520 MHz
QICK running on ZCU216, software version 0.2.422

Firmware configuration (built Mon Jul 27 03:16:23 2026):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 2, DAC tile 3], [ADC tile 2]

	16 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		3

In [4]:
# Jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 # still necessary even after importing all from QICK

# useful libraries
import numpy as np
import random 
import time
import math

In [16]:
RO_CH = 0
f_dds = 430.08 # MHz

## Main Code

In [8]:
from collections import defaultdict

def decompose_schedule(pulse_schedule):
    """
    Decomposes schedule into non-overlapping segments with masks.
    Handles simultaneous tones and repeated frequencies on the same channel.
    All times in µs.
    """
    
    ch_pulses = defaultdict(list)
    for tone in pulse_schedule:
        ch_pulses[tone["channel"]].append(tone)

    decomposed = []
    for ch, tones in ch_pulses.items():
        # Collect all time boundaries
        time_points = set()
        for tone in tones:
            time_points.add(tone["t_off"])
            time_points.add(tone["t_off"] + tone["pulse_length"])
        time_points = sorted(time_points)

        # For each segment find active tones
        for i in range(len(time_points) - 1):
            t_start = time_points[i]
            t_end   = time_points[i + 1]
            active  = [
                tone for tone in tones
                if tone["t_off"] <= t_start and tone["t_off"] + tone["pulse_length"] >= t_end
            ]
            if active:
                decomposed.append({
                    "channel":      ch,
                    "t_off":        t_start,
                    "pulse_length": t_end - t_start,
                    "active_tones": active,
                })

    decomposed.sort(key=lambda p: (p["channel"], p["t_off"]))
    return decomposed

In [17]:
class ScheduledMixMuxProg(AveragerProgramV2):
    def _initialize(self, cfg):
        ch_tones = {}
        for tone in cfg["pulse_schedule"]:
            ch = tone["channel"]
            if ch not in ch_tones:
                ch_tones[ch] = {}
            freq = tone["freq"]
            if freq not in ch_tones[ch]:
                ch_tones[ch][freq] = (tone["gain"], tone["phase"])

        self.ch_freq_index = {}
        for ch, tones in ch_tones.items():
            freqs  = list(tones.keys())
            gains  = [tones[f][0] for f in freqs]
            phases = [tones[f][1] for f in freqs]
            self.ch_freq_index[ch] = {f: i for i, f in enumerate(freqs)}
            
            
            if ((max(freqs) - min(freqs)) < f_dds):
                mixer_freq = (max(freqs) + min(freqs)) / 2  # center the DDS window
            else:
                raise("Range of frequencies to wide")

            self.declare_gen(
                ch=ch, nqz=1, ro_ch=cfg["ro_ch"],
                mixer_freq=mixer_freq,
                mux_freqs=freqs,
                mux_gains=gains,
                mux_phases=phases
            )
        
        # Dummy readout
        self.declare_readout(ch=cfg["ro_ch"], length=cfg["ro_len"])  # ← no gen_ch
        first = cfg["pulse_schedule"][0]
        self.add_readoutconfig(
            ch=cfg["ro_ch"], name="myro",
            freq=first["freq"], phase=0,
            gen_ch=first["channel"]
        )
        
        self.decomposed = decompose_schedule(cfg["pulse_schedule"])
        for i, seg in enumerate(self.decomposed):
            ch    = seg["channel"]
            freqs = [t["freq"] for t in seg["active_tones"]]
            mask  = [self.ch_freq_index[ch][f] for f in freqs]
            self.add_pulse(
                ch=ch,
                name=f"pulse_{i}",
                style="const",
                length=seg["pulse_length"],
                mask=mask
            )

    def _body(self, cfg):
        self.trigger(ros=[cfg["ro_ch"]], pins=[0], t=cfg["trig_time"]) # trigger dummy readout
        
        # Fire pulses
        for i, seg in enumerate(self.decomposed):
            self.pulse(ch=seg["channel"], name=f"pulse_{i}", t=seg["t_off"])

In [19]:
# Note that this code currently does not support two different pulses with the same frequency but different gains on the same channel
config = {
    "ro_ch":     0,
    "ro_len":    1.9,
    "trig_time": 0.4,

    # Pulse schedule, freq (MHz), time (μs)
    "pulse_schedule": [
        {"channel": 0, "freq": 0, "gain": 1.0, "phase": 0, "pulse_length": 800.0, "t_off": 100.0},

        {"channel": 1, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": 150.0, "t_off": 100.0},
        {"channel": 1, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": 150.0, "t_off": 750.0},

        {"channel": 2, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": 800.0, "t_off": 100.0},

        {"channel": 3, "freq": 175, "gain": 1.0, "phase": 0, "pulse_length": 150.0, "t_off": 100.0},
        {"channel": 3, "freq": 175, "gain": 1.0, "phase": 0, "pulse_length": 150.0, "t_off": 750.0},
        {"channel": 3, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": 800.0, "t_off": 100.0},
    ],
}
            
# --- Execution ---
for i in range(1000):
    prog = ScheduledMixMuxProg(soccfg, reps=1000, final_delay=0.0, cfg=config)
    results = prog.acquire(soc) 
    print("Done")

  0%|          | 0/1000 [00:00<?, ?it/s]

clearing streamer buffer
buffer cleared
Done


  0%|          | 0/1000 [00:00<?, ?it/s]

Done


  0%|          | 0/1000 [00:00<?, ?it/s]

...

Done


  0%|          | 0/1000 [00:00<?, ?it/s]

Done


  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
soc.reset_gens()

Note: The max frequency is f_dds / 2, or ~215 MHz, but can be shifted by using a mixer, though currently not used, so long as things are within a range of ~430 MHz.

## Run with Different Wait Times

In [ ]:
# Not 4 ns increments, minimum time between pulses is ~8ns, so multiples of 8 ns
# The reason < 8 ns won't work for this generator is this error:
# RuntimeError: Pulse length of 2 cycles is out of range (exceeds 32 bits, or less than 3)

for i in range(100):
    wait_times = [0, 1000, 2000, ] 

    for j, wait_time in enumerate(wait_times):
        half_length = ((8000 - wait_time) / 2.0) / 1000.0  # µs
        t_first     = 1000.0 / 1000.0                     # µs
        t_second    = (1000.0 + (8000 - wait_time) / 2.0 + wait_time) / 1000.0  # µs
        long_pulse_length = 8000.0/1000.0

        config = {
            "ro_ch":     0,
            "ro_len":    1.9,
            "trig_time": 0.4,
            "pulse_schedule": [
                {"channel": 0, "freq": 0,   "gain": 1.0, "phase": 0, "pulse_length": long_pulse_length, "t_off": t_first},

                {"channel": 1, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": half_length, "t_off": t_first},
                {"channel": 1, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": half_length, "t_off": t_second},

                {"channel": 2, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": long_pulse_length, "t_off": t_first},

                {"channel": 3, "freq": 175, "gain": 1.0, "phase": 0, "pulse_length": half_length, "t_off": t_first},
                {"channel": 3, "freq": 175, "gain": 1.0, "phase": 0, "pulse_length": half_length, "t_off": t_second},
                {"channel": 3, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": long_pulse_length, "t_off": t_first},
            ],
        }

        print(f"Loop {i}, wait_time {j}: wait_time={wait_time}ns, half_length={half_length*1000:.1f}ns, "
              f"t_first={t_first*1000:.1f}ns, t_second={t_second*1000:.1f}ns")

        prog = ScheduledMixMuxProg(soccfg, reps=500, final_delay=0.0, cfg=config)
        results = prog.acquire(soc)
        print("Done")

# Very small wait_times may give warnings but they are likely just warnings not issues
# If we want phrst at some point we'd need to add it to the firmware

In [ ]:
soc.reset_gens()